<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Names and Remapping

The previous notebook connected incoming messages to the subscriber's callback and used graph inspection to explain silence when its publisher stopped. As an application grows, the same implementation may need to run several times, in the namespace of a given robot, or connect to differently named topics.

ROS 2 **remapping** supplies replacement names when a program starts, allowing publishers and subscribers to be redirected without rebuilding the code. This notebook gives a second publisher its own node name, moves a publisher-subscriber exchange to another topic, puts a node in a robot's namespace from the command line, and finally uses remapping to connect `drive_node` to a robot whose kinematics node reads a different topic.

## Prepare each terminal

Prepare three terminals with a router reachable (notebook 03) and earlier exercise nodes stopped. Terminals 1 and 2 run nodes; Terminal 3 is for inspection. The package built in the previous notebooks is reused directly, since every change below is supplied through start-up arguments.

## Give a second publisher a distinct node name

Running another instance of the publisher would create another node named `my_publisher_node`. Although ROS 2 permits duplicate node names, a distinct name allows node-specific queries to identify the second instance unambiguously. Start the first publisher in Terminal 1:

```bash
ros2 run ros2_basics my_publisher_node
```

For the second instance, `--ros-args` introduces ROS-specific arguments, and `-r` supplies a remapping rule. The special name `__node` selects the node's name, so `__node:=my_publisher_two` requests `my_publisher_two` for this instance. In Terminal 2:

```bash
ros2 run ros2_basics my_publisher_node --ros-args -r __node:=my_publisher_two
```

The second terminal prints lines such as:

```shell
[INFO] [1793450200.283584980] [my_publisher_two]: publishing: "Hello from ROS 2! [3]"
```

The logger name reflects the remapped node. While the two publishers keep running, in Terminal 3:

```bash
ros2 node list --no-daemon --spin-time 5
```

```shell
/my_publisher_node
/my_publisher_two
```

Both instances are identifiable, but where do their messages go?

```bash
ros2 topic info /chatter --no-daemon --spin-time 5
```

```shell
Type: std_msgs/msg/String
Publisher count: 2
Subscription count: 0
```

Both publishers still use `/chatter`, because their relative topic name `chatter` resolves under the unchanged root namespace. The node-name remap gives this instance a distinct identity while preserving its connection to the shared topic. Stop both publishers with `Ctrl-C` before continuing.

## Remap both ends of a topic

Suppose the application should exchange its text on `/greeting`. The rule `chatter:=greeting` replaces the topic name used by the node receiving that start-up argument. Start the publisher with this rule in Terminal 1:

```bash
ros2 run ros2_basics my_publisher_node --ros-args -r chatter:=greeting
```

Its log still shows the familiar payload under `my_publisher_node`, since the topic remap changes where the message is published, not what the node prints. Start the subscriber with its usual command in Terminal 2:

```bash
ros2 run ros2_basics my_subscriber_node
```

The subscriber remains running without printing `I heard:` lines. Unlike the stopped-publisher example of notebook 10, publication is continuing, so the endpoint names are the next useful check. In Terminal 3:

```bash
ros2 node info /my_publisher_node
ros2 node info /my_subscriber_node
```

The publisher's output includes `Publishers: /greeting: std_msgs/msg/String`; the subscriber's includes `Subscribers: /chatter: std_msgs/msg/String`. Although the message types agree, the endpoints refer to different topics because only the publisher received the remapping rule. Stop the subscriber in Terminal 2, then restart it with the matching rule:

```bash
ros2 run ros2_basics my_subscriber_node --ros-args -r chatter:=greeting
```

After discovery, its callback begins logging the received messages. An independent observation confirms delivery on the chosen topic, in Terminal 3:

```bash
ros2 topic echo --once --timeout 10 /greeting std_msgs/msg/String
ros2 topic info --verbose /greeting --no-daemon --spin-time 5
```

The verbose output identifies one publisher owned by `my_publisher_node` and one subscription owned by `my_subscriber_node`, both on `/greeting`. The rules apply to the processes started with those arguments: starting the programs again without the rules uses their source-defined names, so the command line becomes part of the configuration needed to reproduce an application's graph. Stop both nodes.

## Put a node in a namespace

A namespace prefixes every relative name of a node at once. The special name `__ns` selects it. In Terminal 1:

```bash
ros2 run ros2_basics my_publisher_node --ros-args -r __ns:=/lab
```

In Terminal 3:

```bash
ros2 node list --no-daemon --spin-time 5
ros2 topic list -t --no-daemon --spin-time 5
```

The node is `/lab/my_publisher_node` and its topic `/lab/chatter`. A subscriber started without the namespace would listen on `/chatter` and hear nothing; `ros2 run ros2_basics my_subscriber_node --ros-args -r __ns:=/lab` hears it. This is exactly what the launch files of the package do with `namespace=veh`: `ros2 launch ros2_basics my_publisher_node.launch.py veh:=lab` is equivalent to the `ros2 run` command above, plus the parameters of the yaml file. Stop the publisher.

Parameters can be given on the command line in the same place, with `-p`:

```bash
ros2 run ros2_basics my_publisher_node --ros-args -r __ns:=/lab -p rate_hz:=3.0 -p message:="three per second"
```

## Remapping on the robot: connect `drive_node` to an older stack

Remapping is how one implementation adapts to the environment it is deployed in. The kinematics node of the Duckiebot reads its car commands on `/ROBOTNAME/car_cmd_switch_node/cmd` in current robot images, but on `/ROBOTNAME/car_cmd` in images built before October 2026. `drive_node` publishes on the relative name `car_cmd_switch_node/cmd`; rather than editing its code for an older robot, its launch file exposes the topic as a launch argument and applies it as a remapping:

```python
DeclareLaunchArgument("car_cmd_topic", default_value=EnvironmentVariable("CAR_CMD_TOPIC", default_value="car_cmd_switch_node/cmd"), ...),
Node(..., remappings=[("car_cmd_switch_node/cmd", car_cmd_topic)], ...)
```

With the robot's stack up and a terminal pointed at its router (notebook 08), find out which name your robot uses:

```bash
ros2 node info /ROBOTNAME/kinematics_node
```

The `Subscribers:` section shows either:

```shell
    /ROBOTNAME/car_cmd_switch_node/cmd: duckietown_msgs/msg/Twist2DStamped
```

in which case the default launch is correct, or:

```shell
    /ROBOTNAME/car_cmd: duckietown_msgs/msg/Twist2DStamped
```

in which case launch the drive node with the remapping:

```bash
ros2 launch ros2_basics drive_node.launch.py veh:=ROBOTNAME car_cmd_topic:=car_cmd
```

and verify the result with `ros2 node info /ROBOTNAME/drive_node`: its publisher is now `/ROBOTNAME/car_cmd`, the name the kinematics node reads, and `ros2 service call /ROBOTNAME/drive_node/start std_srvs/srv/Trigger` moves the robot. For `dts code workbench`, the recipe's launcher reads the same choice from the environment variable `CAR_CMD_TOPIC`.

Distinct node names made two instances individually identifiable, matching topic remaps moved a working exchange to `/greeting`, a namespace remap reproduced what the launch files do, and a launch-file remapping connected the same node to two generations of the robot's stack. Inspecting deliberately mismatched endpoints explained a subscriber's silence and showed how start-up configuration determines the connections between reusable implementations.

## Further reading

The ROS 2 design article on [remapping names](https://design.ros2.org/articles/static_remapping.html) explains the rule syntax, and the [launch tutorials](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Launch/Launch-Main.html) show remappings and parameters in launch files.

Let's move to [the last notebook about Duckietown and ROS 2](../notebooks/12_duckietown_ros2.ipynb).